In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import os
import openpyxl
from pandas import *

In [2]:
dt = 1 / 24
nyears = 1000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Marked Poisson
rain = md.poisson_process(n = int(nsim), alpha = 12.2/1000, lbd = (0.22 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) # Hour

days = np.arange(365)
annual_avg_npp = 737.3

shape = np.sin((np.pi*days)/365)                       

daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [3]:
print("Maximum Value is:", max(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 3.1729987858208033
Average value is: 2.019999999999999


In [4]:
total_rainfall = rain.sum()                  # total rainfall in 'm'
print(f"Total rainfall over 2000 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year in 'mm'
print(f"Average rainfall per year = {RPY:.3f} mm/year")

PET = 1303.6
AI = PET/RPY
print(AI)

Total rainfall over 2000 years = 981.818 meters
Average rainfall per year = 981.818 mm/year
1.3277415721401442


In [5]:
# Compute TI_DI and TI_Texture

KA_final = 9.0132
n_final = 1.7444
R_max = 3.8                   

def compute_TI(AI):
    AI = np.asarray(AI, dtype=float)
    numerator   = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor

    return np.minimum(TI, 1.0)

# ------------------------------------
# TI FROM SAND (SIGMOID)
# ------------------------------------
sand = 47.0   # DOU sand (%)
s50 = 25.0
k   = 0.08

def TI_from_sand(sand):
    return 1.0 / (1.0 + np.exp(-k * (sand - s50)))

TI_sand = TI_from_sand(sand)
TI_AI = compute_TI(AI)
TI = (TI_sand + TI_AI)/2

print("TI_DI =", TI_AI)
print("TI_tex =", TI_sand)
print("TI Total =", TI)

TI_DI = 0.1884621156072825
TI_tex = 0.8532096601986177
TI Total = 0.5208358879029501


In [6]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.15   #m/d,
kstot = 1     #m/d
phitex = 0.50  
phitot = 0.60  

In [7]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████████████████████████████████████████████████████████████████| 8760000/8760000 [18:55<00:00, 7711.51it/s]


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       942.050163   52.705194   45.059706  0.244376  0.005813  0.398565   
2       972.400193    7.635421   78.974527  0.264007  0.002126  0.057738   
3      1087.999927    2.124440  147.240452  0.296926  0.000861  0.016063   
4      1162.614827    1.933944  261.699828  0.321239  0.000923  0.014620   
5      1178.568219   11.617450  408.200977  0.298494  0.003488  0.087804   
...            ...         ...         ...       ...       ...       ...   
996    1776.026616  135.110762  192.509728  0.484062  0.015278  1.021496   
997    1775.519695  135.308456  200.666496  0.470032  0.014829  1.022973   
998    1775.419061  139.556212  190.536852  0.472837  0.014993  1.055099   
999    1775.272740  138.297695  188.091199  0.454810  0.014405  1.045586   
1000   1776.295558  138.965411  190.644398  0.478656  0.015170  1.050634   

           

In [8]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Dou\DOU.csv", index= True, header = True)
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\DOU.csv", index= True, header = True)